In this project, I will be building a toy transformer.

## Implement multi-head attention from scratch

First, we will implement a multi-head attention mechanism. Recall that the input to the attention mechanism is batched sequences of word vectors. So it's shape should be (batch_size, seq_length, n_hidden) where n_hidden is the size of a word vector (or output vector size from the previous layer).

Inside the multi-head attention, we have the following steps.
1. Create query, key, value vectors for every position in the sequence. This is simply done by dot products with our three parameter matrices whose sizes are exactly the same.
2. Split each vector into n_heads equally sized parts. The attention heads operate in parallel, so here we extract the parts of query, key, and value vectors that belong to each attention head. The vectors now have size n_hidden / n_heads (we set up these two parameters such that they are dividable). The order of the first two steps can be swapped if you set the output size of query, key, value parameters to n_hidden / n_heads directly, but this way is more efficient.
3. Compute the attention matrix. This is done by taking the dot product between every query and every key. Because we are doing self-attention, we have the same number of queries and keys, so the attention matrix is a square matrix. However, it's important to still pay attention to the dimension names in einsum, since (query_index, key_index) is not the same as (key_index, query_index). After the dot product, we normalize the attention matrix along the dimension that corresponds to the keys.
4. Compute the weighted average of values. For every query, compute the average of all value vectors weighted according to the attention matrix. In this step, the indices you use to query the attention matrix need to be consistent with the indices you used in the previous step.
5. Aggregate the attention heads and compute the output. Here we just concatenate the results from all attention heads together, and do a final dot product to get the final output.

In [1]:
import math
import torch
from torch import nn
from einops import rearrange

In [3]:
class MultiHeadAttention(nn.Module):

    def __init__(self, n_heads, n_hidden):
        super().__init__()
        self.n_heads = n_heads
        # all the hidden vectors will have the same size
        # the hidden dimension of each attention head is n_hidden / n_heads
        self.W_Q = nn.Parameter(torch.rand((n_hidden, n_hidden)))
        self.W_K = nn.Parameter(torch.rand((n_hidden, n_hidden)))
        self.W_V = nn.Parameter(torch.rand((n_hidden, n_hidden)))
        self.W_O = nn.Parameter(torch.rand((n_hidden, n_hidden)))

    def forward(self, X):
        # X shape: batch_size, length, n_hidden

        # Step 1: compute query, key, value vectors
        Q = torch.einsum('blh,hk->blk', X, self.W_Q)
        K = torch.einsum('blh,hk->blk', X, self.W_K)
        V = torch.einsum('blh,hk->blk', X, self.W_V)

        # Step 2: split into n_heads equal sized parts
        Q = rearrange(Q, 'b l (n d) -> b n l d', n=self.n_heads)
        K = rearrange(K, 'b l (n d) -> b n l d', n=self.n_heads)
        V = rearrange(V, 'b l (n d) -> b n l d', n=self.n_heads)

        # Step 3: compute attention matrix
        # This is the most tricky part
        # Make sure that the dimensions are: batch, n_heads, *n_queries*, *n_keys*
        # not batch, n-Heads, *n_keys*, *n_queries*
        A = torch.einsum('bnqd,bnkd->bnqk', Q, K)

        # Normalize the attention matrix
        A = A / math.sqrt(Q.shape[-1])
        A = A.softmax(dim=-1)

        # Step 4: compute the weighted average
        # Make sure the dimensions are lined up with how you compute the matrix
        # Make sure that the output shape is batch, n_heads, *n_queries*, n_hidden
        F = torch.einsum('bnqk,bnkd->bnqd', A, V)

        # Step 5: aggregate attention heads and compute the output
        F = rearrange(F, 'b n q d -> b q (n d)')
        F = torch.einsum('b l h, h j -> b l j', F, self.W_O)
        return F

We will also implement LayerNorm from scratch. This is actually easier than BatchNorm which you implemented earlier.

In [4]:
class LayerNorm(nn.Module):

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps # small number to make sure sqrt doesn't run into numerical issues
        # two learned parameters for scale and shift, just like batch norm
        self.gamma = nn.Parameter(torch.ones(dim))  # Scale
        self.beta = nn.Parameter(torch.zeros(dim))  # Shift

    def forward(self, X):
        # X shape: (batch, length, dim)
        mean = X.mean(dim=-1, keepdim=True)
        var = X.var(dim=-1, unbiased=False, keepdim=True)
        # normalize the whole tensor using mean and variance computed above
        x_norm = (X - mean) / torch.sqrt(var + self.eps)
        # apply scale and norm
        return self.gamma * x_norm + self.beta

## Build a Transformer block
Recall that a Transformer block is just a multi-head attention followed by a feed-forward layer, with layer normalization and skip connection applied to both. A skip connection for a layer simply means you take the output of that layer, and add it up with the input, and this is the final output. In a Transformer block, the result of this skip connection then goes through normalization.

In [5]:
class TransformerBlock(nn.Module):

    def __init__(self, n_heads, n_hidden):
        super().__init__()
        self.attn = MultiHeadAttention(n_heads, n_hidden)
        # separate layer normalizations for attention and feed-forward
        # since the scale and shift for these two layers should be different
        self.norm1 = LayerNorm(n_hidden)
        self.norm2 = LayerNorm(n_hidden)
        # this is a default choice for the size of the feed-foward layer
        ff_hidden_dim = n_hidden * 4

        # the feed-forward layer uses ReLU
        self.ff = nn.Sequential(
            nn.Linear(n_hidden, ff_hidden_dim),
            nn.ReLU(),
            nn.Linear(ff_hidden_dim, n_hidden)
        )

    def forward(self, X):
        # Attention with residual & norm
        # Compute the output of attention layer
        attn_out = self.attn(X)
        # Compute the skip connection, then send it through the first layer norm
        attn_out = self.norm1(X + attn_out)

        # Feedforward with residual & norm
        # Compute the output of ff layer
        ff_out = self.ff(attn_out)
        # Compute the skip connection, then send it through the second layer norm
        ff_out = self.norm2(attn_out + ff_out)
        return ff_out

Now that we have a Transformer block, a Transformer model becomes super simple. A toy Transformer model with one block is just an embedding layer, the block, and an output layer.

In [6]:
class ToyTransformerModel(nn.Module):
    def __init__(self, vocab_size, seq_len, n_hidden=32, n_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, n_hidden)
        self.block = TransformerBlock(n_heads, n_hidden)
        self.output = nn.Linear(n_hidden, vocab_size)

    def forward(self, x):
        x = self.embed(x)  # (batch, seq_len, n_hidden)
        x = self.block(x)
        logits = self.output(x)  # (batch, seq_len, vocab_size)
        return logits

You can use the following synthetic training setup to verify that your code will run. Convergence doesn't necessarily mean that you've implemented the model correctly. We will still carefully check if your implementation is actually correct, even if this part works fine.

In [7]:
def generate_data(batch_size, seq_len, vocab_size):
    X = torch.randint(0, vocab_size, (batch_size, seq_len))
    Y = torch.roll(X, shifts=-1, dims=1)
    return X, Y


vocab_size = 20
seq_len = 8
model = ToyTransformerModel(vocab_size, seq_len)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


for epoch in range(400):
    model.train()
    X, Y = generate_data(batch_size=32, seq_len=seq_len, vocab_size=vocab_size)
    logits = model(X)
    loss = criterion(logits.view(-1, vocab_size), Y.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 3.2037
Epoch 10 | Loss: 3.0698
Epoch 20 | Loss: 3.0062
Epoch 30 | Loss: 2.9795
Epoch 40 | Loss: 2.9635
Epoch 50 | Loss: 2.9588
Epoch 60 | Loss: 2.9866
Epoch 70 | Loss: 2.9538
Epoch 80 | Loss: 2.9408
Epoch 90 | Loss: 2.9114
Epoch 100 | Loss: 2.9755
Epoch 110 | Loss: 2.8954
Epoch 120 | Loss: 2.9222
Epoch 130 | Loss: 2.9044
Epoch 140 | Loss: 2.8524
Epoch 150 | Loss: 2.9040
Epoch 160 | Loss: 2.8822
Epoch 170 | Loss: 2.8789
Epoch 180 | Loss: 2.8645
Epoch 190 | Loss: 2.8556
Epoch 200 | Loss: 2.8008
Epoch 210 | Loss: 2.8396
Epoch 220 | Loss: 2.8404
Epoch 230 | Loss: 2.7876
Epoch 240 | Loss: 2.7982
Epoch 250 | Loss: 2.8306
Epoch 260 | Loss: 2.8316
Epoch 270 | Loss: 2.7948
Epoch 280 | Loss: 2.8229
Epoch 290 | Loss: 2.8228
Epoch 300 | Loss: 2.7997
Epoch 310 | Loss: 2.8459
Epoch 320 | Loss: 2.7755
Epoch 330 | Loss: 2.8044
Epoch 340 | Loss: 2.7958
Epoch 350 | Loss: 2.8168
Epoch 360 | Loss: 2.7414
Epoch 370 | Loss: 2.7079
Epoch 380 | Loss: 2.7934
Epoch 390 | Loss: 2.7086
